# Michigan Traders: Module 5
# Indicators, Signals & Strategy Design Patterns  ·  *ANSWER KEY*

**Series:** MAT Education · QuantConnect Core
**Level:** Intermediate (builds on Modules 1–4)
**Format:** **Guided + Your-Turn**, with self-checking exercises you can run on your own laptop.

---

Module 3 introduced indicators as one of the five pillars and used two of them. Module 4 showed you how to pull them over a whole history in research. This module makes them the main subject.

Two things are going on whenever you use an indicator, and beginners routinely confuse them:

1. **The indicator**: a *stateful* transformation of a price stream. It remembers past bars and produces one number per bar. `SimpleMovingAverage(20)` is an indicator.
2. **The signal**: a *decision rule* built on indicator values. "Go long when the 20-day crosses above the 50-day" is a signal.

Indicators are arithmetic; there is exactly one right answer. Signals are strategy design, where all the judgement lives. This module teaches the arithmetic first so that you can then focus on the judgement.

## How to use this notebook

Same two-environment split as Module 4:

| Cell type | Where it runs | What to do |
|---|---|---|
| 🟢 **Local cell** | your laptop's Jupyter | Run it. Output is baked in so you can read along. |
| 🔵 **QC cell** | QuantConnect (LEAN) | Copy into an algorithm project. It will *not* run locally. |

Every indicator in this module is taught **twice**: once as the QuantConnect call you will actually write, and once as the pandas equivalent you can compute and check locally. That pairing is deliberate. When a backtest misbehaves, being able to reproduce an indicator by hand in a research notebook is how you find out whether the bug is in your signal or in your understanding.

Answers are in `05_Indicators_Signals_and_Strategy_Patterns_SOLUTIONS.ipynb`.

### Setup: a price series with regimes

Module 4's panel was a pure random walk, which is fine for correlation work but poor for studying trend indicators: a driftless random walk produces very few clean crossovers.

So this notebook's data has **four regimes** stitched together: up, chop, down, up. That gives moving averages something to cross and breakouts something to break, which is what you need in order to *see* an indicator working.

We build full OHLC bars, because ATR and other range-based indicators need highs and lows, not just closes.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)

rng = np.random.default_rng(11)
dates = pd.bdate_range("2022-01-03", periods=504)

# Four stitched regimes: (n_days, daily drift, daily vol)
regimes = [(126, 0.0012, 0.009),    # steady uptrend
           (126, 0.0000, 0.013),    # choppy, no direction
           (126, -0.0011, 0.012),   # downtrend
           (126, 0.0009, 0.008)]    # recovery

daily = np.concatenate([rng.normal(mu, sd, n) for n, mu, sd in regimes])
close = 100 * np.exp(np.cumsum(daily))

# Give each bar a plausible high/low/open around its close.
wiggle = np.abs(rng.normal(0.0, 0.006, len(dates)))
bars = pd.DataFrame({
    "open":  close * (1 + rng.normal(0.0, 0.003, len(dates))),
    "high":  close * (1 + wiggle),
    "low":   close * (1 - wiggle),
    "close": close,
}, index=dates)
bars.index.name = "time"

px = bars["close"]        # shorthand we will use throughout

print("regimes:", [n for n, _, _ in regimes], "days each")
print("price range:", round(px.min(), 2), "->", round(px.max(), 2))
bars.head(3)

regimes: [126, 126, 126, 126] days each
price range: 99.34 -> 135.49


,open,high,low,close
time,,,,
2022-01-03,99.5860,100.3633,99.9385,100.1509
2022-01-04,100.5086,101.6860,101.3256,101.5058
2022-01-05,103.4665,103.2135,102.2945,102.7540


Before touching indicators, look at the shape of the series. The four regimes should be visible as distinct slopes in the running level.

In [2]:
checkpoints = [0, 125, 251, 377, 503]
levels = pd.DataFrame({
    "day": checkpoints,
    "date": [px.index[i].date() for i in checkpoints],
    "close": [round(float(px.iloc[i]), 2) for i in checkpoints],
})
levels

,day,date,close
0,0,2022-01-03,100.15
1,125,2022-06-27,121.57
2,251,2022-12-20,119.33
3,377,2023-06-14,119.77
4,503,2023-12-07,131.66


Up from ~100 to ~115, sideways, down to ~100, then back up. Good: a trend follower should make money in regimes 1 and 4, lose in regime 2, and (if it can go short) make money in regime 3.

## 1. What "stateful" means, and why it matters

A pandas rolling mean looks at an entire column at once. A QuantConnect indicator does not: it is fed **one bar at a time**, updates its internal state, and exposes its current value.

```
bar 1 ──► indicator.update(bar) ──► internal state ──► .current.value
bar 2 ──► indicator.update(bar) ──► internal state ──► .current.value
...
```

Three consequences follow, and all three cause real bugs:

1. **An indicator is not ready immediately.** A 20-day SMA needs 20 bars before it means anything. Until then `.is_ready` is `False` and its value is garbage. Section 4 covers this.
2. **It cannot see the future.** This is a *feature*: it is structurally impossible to leak tomorrow's price into today's signal, which is not true of the pandas approach.
3. **Order matters.** Feeding bars out of order corrupts the state.

In research (Module 4) you had the whole series in memory and used `.rolling()`. In an algorithm you use indicator objects. They agree on the answer, but only the second one is safe to trade.

## 2. Automatic indicators: the helper methods

The easy path, and the one you will use 90% of the time. `QCAlgorithm` exposes a lowercase helper for every common indicator. Call it in `initialize` and QuantConnect wires up the updates for you.

```python
self.sma_20 = self.sma("SPY", 20, Resolution.DAILY)
              │        │      │   └── resolution of the bars feeding it
              │        │      └────── period (lookback in bars)
              │        └───────────── ticker or Symbol
              └────────────────────── helper name = indicator name, lowercase
```

You saw `self.sma` and `self.rsi` in Module 3. Here is the wider set you will actually reach for:

| Helper | Indicator | What it measures |
|---|---|---|
| `self.sma(sym, n)` | Simple Moving Average | average price over `n` bars |
| `self.ema(sym, n)` | Exponential Moving Average | weighted average, recent bars count more |
| `self.rsi(sym, n)` | Relative Strength Index | momentum, 0–100, overbought/oversold |
| `self.bb(sym, n, k)` | Bollinger Bands | a moving average plus `k` standard-deviation bands |
| `self.atr(sym, n)` | Average True Range | typical bar range, a volatility measure in price units |
| `self.macd(sym, f, s, sig)` | MACD | difference of two EMAs, plus a signal line |
| `self.std(sym, n)` | Standard Deviation | volatility of price over `n` bars |
| `self.mom(sym, n)` | Momentum | price now minus price `n` bars ago |
| `self.max(sym, n)` / `self.min(sym, n)` | Rolling extremes | highest high / lowest low, for breakouts |

Each returns an **indicator object**, which you store on `self` so `on_data` can read it later.

In [ ]:
# 🔵 QC cell, registering several automatic indicators
class IndicatorZoo(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)

        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # Each of these is created ONCE, in initialize, and updates itself
        # automatically on every new bar.
        self.sma_20 = self.sma(self.symbol, 20, Resolution.DAILY)
        self.sma_50 = self.sma(self.symbol, 50, Resolution.DAILY)
        self.ema_20 = self.ema(self.symbol, 20, Resolution.DAILY)
        self.rsi_14 = self.rsi(self.symbol, 14, Resolution.DAILY)
        self.atr_14 = self.atr(self.symbol, 14, Resolution.DAILY)
        self.bb_20 = self.bb(self.symbol, 20, 2, Resolution.DAILY)

    def on_data(self, data: Slice):
        pass

> ⚠️ **Create indicators in `initialize`, never in `on_data`.** Creating one inside `on_data` builds a brand-new, empty indicator on every bar, so it never becomes ready and its value is always zero. This is the single most common indicator bug, and it fails silently: no exception, just a strategy that never trades.

## 3. Reading an indicator's value

An indicator object is not a number. To get the number you go through `.current.value`:

```python
self.sma_20.current.value     # today's 20-day average, as a float
```

| Expression | Type | Meaning |
|---|---|---|
| `self.sma_20` | indicator object | the indicator itself |
| `self.sma_20.current` | `IndicatorDataPoint` | the latest point: has `.time` and `.value` |
| `self.sma_20.current.value` | `float` | the number you compare against |
| `self.sma_20.is_ready` | `bool` | has it seen enough bars yet? |

Multi-output indicators expose named sub-indicators instead of a single value:

| Indicator | Sub-values |
|---|---|
| `self.bb_20` | `.middle_band`, `.upper_band`, `.lower_band` (each `.current.value`) |
| `self.macd_x` | `.current.value` (the MACD line), `.signal`, `.histogram` |

In [ ]:
# 🔵 QC cell, reading values inside on_data
class ReadingValues(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.sma_20 = self.sma(self.symbol, 20, Resolution.DAILY)
        self.bb_20 = self.bb(self.symbol, 20, 2, Resolution.DAILY)

    def on_data(self, data: Slice):
        # Pillar 2 guard from Module 3: is there actually a bar for our symbol?
        if not data.bars.contains_key(self.symbol):
            return

        # Indicator guard: is there enough history for the value to be meaningful?
        if not self.sma_20.is_ready or not self.bb_20.is_ready:
            return

        price = data.bars[self.symbol].close
        average = self.sma_20.current.value
        upper = self.bb_20.upper_band.current.value
        lower = self.bb_20.lower_band.current.value

        self.debug(f"price={price:.2f} sma={average:.2f} band=[{lower:.2f}, {upper:.2f}]")

Note the **two guards** stacked at the top of `on_data`. The first (`data.bars.contains_key`) is Module 3's data guard: a bar might be missing on a given day. The second (`.is_ready`) is the indicator guard. You need both, and in that order.

## 4. Warm-up: the fix for `is_ready`

A 50-day moving average is not ready until day 50. If your backtest starts on 2022-01-01, you either waste the first 50 days doing nothing, or you ask QuantConnect to feed the indicator **historical data from before the start date**. The second option is `set_warm_up`.

```python
self.set_warm_up(50, Resolution.DAILY)   # feed 50 daily bars before day one
```

During warm-up, `on_data` still fires, so you guard with `self.is_warming_up` to avoid trading on pre-start data.

In [ ]:
# 🔵 QC cell, warm-up done properly
class WarmedUp(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        self.sma_50 = self.sma(self.symbol, 50, Resolution.DAILY)

        # Ask for slightly MORE than the longest lookback, so everything is
        # ready on the very first tradable bar.
        self.set_warm_up(60, Resolution.DAILY)

    def on_data(self, data: Slice):
        if self.is_warming_up:
            return                      # indicators are updating; do not trade yet
        if not self.sma_50.is_ready:
            return
        if not data.bars.contains_key(self.symbol):
            return

        # ... real logic here ...

Rules of thumb:

- Warm up to **the longest lookback in your algorithm**, plus a small buffer. Two indicators of 20 and 200 bars means warming up ~210 bars.
- `self.is_warming_up` is `True` for the whole warm-up period; check it *first* in `on_data`.
- Without warm-up your strategy silently does nothing for the first N bars, and a 200-day filter can eat most of a one-year backtest.

### ✏️ Your turn: write initialize with warm-up

Write the `initialize` method for an algorithm that:

- runs **2021-01-01** to **2024-01-01** with **$250,000**
- trades **QQQ** at daily resolution, symbol stored on `self.symbol`
- creates a **10-bar EMA** (`self.fast`), a **100-bar SMA** (`self.slow`) and a **14-bar ATR** (`self.atr_14`)
- warms up enough bars for all three to be ready on the first tradable bar

This is a 🔵 QC cell, so there is no self-check. Compare it with the answer key.

In [ ]:
# 🔵 QC cell, no local self-check for this one
class WarmUpPractice(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2021, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(250_000)

        self.symbol = self.add_equity("QQQ", Resolution.DAILY).symbol

        self.fast = self.ema(self.symbol, 10, Resolution.DAILY)
        self.slow = self.sma(self.symbol, 100, Resolution.DAILY)
        self.atr_14 = self.atr(self.symbol, 14, Resolution.DAILY)

        # The longest lookback is 100 bars, so warm up a little past it.
        self.set_warm_up(110, Resolution.DAILY)

    def on_data(self, data: Slice):
        if self.is_warming_up:
            return

## 5. Manual indicators and `register_indicator`

Sometimes the helper is not enough. You want an indicator on a *derived* stream (a spread between two stocks, say, which is exactly what Module 7 does), or on a custom bar frequency.

Then you construct the indicator yourself and take responsibility for updating it.

| Step | Automatic (`self.sma`) | Manual |
|---|---|---|
| Create | `self.sma(sym, 20)` | `SimpleMovingAverage(20)` |
| Update | QuantConnect does it | `ind.update(time, value)`. You do it |
| When to use | indicator on one subscribed security | indicator on anything else |

`update(time, value)` takes a timestamp and a float. There is also `register_indicator`, a middle path: you build the object yourself but ask QuantConnect to keep feeding it a subscribed security's bars.

In [ ]:
# 🔵 QC cell, manual updates on a derived series
from QuantConnect.Indicators import SimpleMovingAverage, StandardDeviation

class ManualIndicators(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_cash(100_000)
        self.a = self.add_equity("KO", Resolution.DAILY).symbol
        self.b = self.add_equity("PEP", Resolution.DAILY).symbol

        # An indicator on the PRICE RATIO between two stocks. No helper exists
        # for this, because the ratio is not a subscribed security.
        self.ratio_sma = SimpleMovingAverage(20)
        self.ratio_std = StandardDeviation(20)

    def on_data(self, data: Slice):
        if not (data.bars.contains_key(self.a) and data.bars.contains_key(self.b)):
            return

        ratio = data.bars[self.a].close / data.bars[self.b].close

        # We feed them ourselves, once per bar, with the bar's timestamp.
        self.ratio_sma.update(self.time, ratio)
        self.ratio_std.update(self.time, ratio)

        if not self.ratio_sma.is_ready:
            return

        z = (ratio - self.ratio_sma.current.value) / self.ratio_std.current.value
        self.debug(f"ratio={ratio:.4f} z-score={z:.2f}")

`self.time` is the algorithm's current timestamp, the "now" of the backtest. You will use it whenever you update an indicator manually or log something.

The alternative, when your derived stream is really just a subscribed security at a different resolution:

In [ ]:
# 🔵 QC cell, register_indicator: you build it, QC feeds it
from QuantConnect.Indicators import RelativeStrengthIndex

class Registered(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        self.rsi_21 = RelativeStrengthIndex(21)
        self.register_indicator(self.symbol, self.rsi_21, Resolution.DAILY)
        # From here on QuantConnect updates rsi_21 automatically, exactly as if
        # it had come from self.rsi(...).

## 6. The indicator catalogue, twice over

Now the arithmetic. For each indicator: what it measures, the QuantConnect call, and the pandas equivalent you can verify locally.

### 6.1 Simple Moving Average

The average of the last `n` closes. It smooths noise and defines "the trend" for most systems.

- **QC:** `self.sma(symbol, 20)`
- **pandas:** `close.rolling(20).mean()`

In [3]:
sma_20 = px.rolling(20).mean()
sma_50 = px.rolling(50).mean()

frame = pd.DataFrame({"close": px, "sma_20": sma_20, "sma_50": sma_50})
print("first 19 rows of sma_20 are NaN:", bool(frame["sma_20"].head(19).isna().all()))
frame.iloc[[18, 19, 20, -1]]

first 19 rows of sma_20 are NaN: True


,close,sma_20,sma_50
time,,,
2022-01-27,105.3555,NaN,NaN
2022-01-28,106.1350,103.3736,NaN
2022-01-31,105.4333,103.6377,NaN
2023-12-07,131.6581,132.7284,132.3592


Row 19 (the 20th bar) is the first non-`NaN` value. That is `is_ready` becoming `True`, expressed in pandas.

### 6.2 Exponential Moving Average

A weighted average where recent bars matter more. It reacts faster than an SMA of the same period, at the cost of being noisier.

The weight decays by a factor `α = 2 / (n + 1)` per bar.

- **QC:** `self.ema(symbol, 20)`
- **pandas:** `close.ewm(span=20, adjust=False).mean()`

`adjust=False` is the important argument: it makes pandas use the same recursive formula QuantConnect does, `ema_today = α × price + (1 − α) × ema_yesterday`. With `adjust=True` (the pandas default) you get a slightly different series that will not match your backtest.

In [4]:
ema_20 = px.ewm(span=20, adjust=False).mean()
alpha = 2 / (20 + 1)

# Verify the recursion by hand on one bar:
manual = alpha * px.iloc[25] + (1 - alpha) * ema_20.iloc[24]
print("alpha            :", round(alpha, 4))
print("pandas ema[25]   :", round(float(ema_20.iloc[25]), 4))
print("hand-computed    :", round(float(manual), 4))
print("match            :", np.isclose(manual, ema_20.iloc[25]))

alpha            : 0.0952
pandas ema[25]   : 103.5932
hand-computed    : 103.5932
match            : True


Compare how the two averages respond when the trend turns. The EMA turns first, which is the whole point of using one.

In [5]:
turn = pd.DataFrame({"close": px, "sma_20": sma_20, "ema_20": ema_20}).iloc[248:254]
turn

,close,sma_20,ema_20
time,,,
2022-12-15,115.7366,115.4348,115.9139
2022-12-16,116.7929,115.6064,115.9977
2022-12-19,118.2303,115.7887,116.2103
2022-12-20,119.3315,115.9006,116.5075
2022-12-21,120.7081,116.0832,116.9076
2022-12-22,120.0135,116.1475,117.2034


### 6.3 Relative Strength Index (RSI)

A momentum oscillator bounded between 0 and 100. It compares the size of recent gains to recent losses:

```
RS  = average gain over n bars / average loss over n bars
RSI = 100 − 100 / (1 + RS)
```

Conventionally, above 70 is "overbought" and below 30 is "oversold". Those thresholds are a starting point, not a law.

- **QC:** `self.rsi(symbol, 14)`

In [6]:
def rsi_simple(prices, n=14):
    delta = prices.diff()
    gain = delta.clip(lower=0)          # positive moves, negatives -> 0
    loss = -delta.clip(upper=0)         # magnitude of negative moves
    avg_gain = gain.rolling(n).mean()
    avg_loss = loss.rolling(n).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)


rsi_14 = rsi_simple(px, 14)

print("RSI range:", round(float(rsi_14.min()), 1), "->", round(float(rsi_14.max()), 1))
print("days overbought (>70):", int((rsi_14 > 70).sum()))
print("days oversold  (<30):", int((rsi_14 < 30).sum()))
rsi_14.dropna().head(3)

RSI range: 20.2 -> 87.7
days overbought (>70): 71
days oversold  (<30): 32


time
2022-01-21    70.8274
2022-01-24    68.0606
2022-01-25    66.6975
Freq: B, Name: close, dtype: float64

`.clip(lower=0)` replaces every negative value with 0, so `gain` holds only up-moves. `-delta.clip(upper=0)` does the mirror image and flips the sign, so `loss` holds the *magnitude* of down-moves. Both are positive series, which is what the RS ratio needs.

> ⚠️ **QuantConnect's `self.rsi` uses Wilder's smoothing by default**, not a simple rolling mean. Wilder's is an EMA with `α = 1/n`. The version above is the textbook definition and is close, but it will not match QC bar for bar. Here is the matching one:

In [7]:
def rsi_wilder(prices, n=14):
    delta = prices.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / n, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / n, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)


compare = pd.DataFrame({"simple": rsi_simple(px), "wilder": rsi_wilder(px)})
compare.iloc[[20, 100, 300, 503]]

,simple,wilder
time,,
2022-01-31,67.8682,81.9059
2022-05-23,63.2610,61.7587
2023-02-27,25.3103,34.6442
2023-12-07,48.4959,44.6604


Same story, slightly different numbers. When you need research and backtest to agree exactly, use the Wilder version, or better, use `qb.indicator(...)` from Module 4, which runs QuantConnect's own implementation.

### 6.4 Bollinger Bands

A moving average with volatility bands drawn `k` standard deviations above and below it (`k` is usually 2). Price near the upper band is stretched relative to its recent range.

- **QC:** `self.bb(symbol, 20, 2)` → `.middle_band`, `.upper_band`, `.lower_band`

In [8]:
n, k = 20, 2
middle = px.rolling(n).mean()
sd = px.rolling(n).std(ddof=0)          # population std, as QuantConnect uses
upper = middle + k * sd
lower = middle - k * sd

bands = pd.DataFrame({"close": px, "lower": lower, "middle": middle, "upper": upper})
inside = ((px > lower) & (px < upper)).mean()
print("share of days inside the bands:", round(float(inside), 3))
bands.tail(3)

share of days inside the bands: 0.855


,close,lower,middle,upper
time,,,,
2023-12-05,133.3713,131.0323,132.9264,134.8206
2023-12-06,132.4120,131.1654,132.7935,134.4216
2023-12-07,131.6581,131.0296,132.7284,134.4272


`ddof=0` asks pandas for the population standard deviation (divide by `n`) instead of the sample one (divide by `n − 1`). QuantConnect's `StandardDeviation` indicator uses the population version, so matching it means passing `ddof=0`. It is a small difference that produces a persistent small mismatch if you get it wrong.

A useful derived quantity is **%B**, which expresses where price sits within the bands on a 0-to-1 scale:

In [9]:
pct_b = (px - lower) / (upper - lower)

print("%B below 0 means price is under the lower band:", int((pct_b < 0).sum()), "days")
print("%B above 1 means price is over the upper band :", int((pct_b > 1).sum()), "days")
pct_b.tail(3)

%B below 0 means price is under the lower band: 18 days
%B above 1 means price is over the upper band : 36 days


time
2023-12-05    0.6174
2023-12-06    0.3828
2023-12-07    0.1850
Freq: B, Name: close, dtype: float64

### ✏️ Your turn: Bollinger Bands and %B

Using `px`, build a DataFrame `bb` with a **30-bar, 1.5-sigma** Bollinger set, four columns in this order:

| column | definition |
|---|---|
| `middle` | 30-bar SMA |
| `upper` | `middle + 1.5 × std` |
| `lower` | `middle − 1.5 × std` |
| `pct_b` | `(close − lower) / (upper − lower)` |

Use the **population** standard deviation (`ddof=0`), the way QuantConnect does. Then set `days_outside` to the number of days `pct_b` is below 0 or above 1.

In [10]:
mid30 = px.rolling(30).mean()
sd30 = px.rolling(30).std(ddof=0)

bb = pd.DataFrame({
    "middle": mid30,
    "upper": mid30 + 1.5 * sd30,
    "lower": mid30 - 1.5 * sd30,
    "pct_b": (px - (mid30 - 1.5 * sd30)) / ((mid30 + 1.5 * sd30) - (mid30 - 1.5 * sd30)),
})

days_outside = int(((bb["pct_b"] < 0) | (bb["pct_b"] > 1)).sum())

print("days outside the bands:", days_outside)
bb.tail(3)

days outside the bands: 154


,middle,upper,lower,pct_b
time,,,,
2023-12-05,133.4261,135.1592,131.6930,0.4842
2023-12-06,133.4221,135.1601,131.6841,0.2094
2023-12-07,133.3408,135.1311,131.5505,0.0301


In [11]:
_m = px.rolling(30).mean()
_s = px.rolling(30).std(ddof=0)
_u, _l = _m + 1.5 * _s, _m - 1.5 * _s
_pb = (px - _l) / (_u - _l)
assert list(bb.columns) == ["middle", "upper", "lower", "pct_b"], \
    f"columns must be middle, upper, lower, pct_b - got {list(bb.columns)}"
assert np.allclose(bb["middle"].dropna().values, _m.dropna().values), "middle should be the 30-bar SMA"
assert np.allclose(bb["upper"].dropna().values, _u.dropna().values), \
    "upper is off - use ddof=0 for the population standard deviation"
assert np.allclose(bb["lower"].dropna().values, _l.dropna().values), "lower is off"
assert np.allclose(bb["pct_b"].dropna().values, _pb.dropna().values), "pct_b is off"
assert days_outside == int(((_pb < 0) | (_pb > 1)).sum()), \
    f"expected {int(((_pb < 0) | (_pb > 1)).sum())} days outside the bands"
print("✅ Correct!  Price closed outside the 1.5-sigma bands on", days_outside,
      "of", int(_pb.notna().sum()), "valid days")

✅ Correct!  Price closed outside the 1.5-sigma bands on 154 of 475 valid days


### 6.5 Average True Range (ATR)

A volatility measure in **price units** rather than percent. True Range for a bar is the largest of:

1. `high − low` (today's range)
2. `|high − previous close|` (gap up)
3. `|low − previous close|` (gap down)

ATR is the average True Range over `n` bars. It is the standard input to volatility-scaled position sizing and stop placement, both of which appear in Module 9.

- **QC:** `self.atr(symbol, 14)`

In [12]:
prev_close = bars["close"].shift(1)
tr = pd.concat([
    bars["high"] - bars["low"],
    (bars["high"] - prev_close).abs(),
    (bars["low"] - prev_close).abs(),
], axis=1).max(axis=1)

atr_14 = tr.rolling(14).mean()

print("ATR in price units:", round(float(atr_14.iloc[-1]), 3))
print("as a % of price   :", f"{float(atr_14.iloc[-1] / px.iloc[-1]):.2%}")
pd.DataFrame({"high": bars["high"], "low": bars["low"], "true_range": tr, "atr_14": atr_14}).tail(3)

ATR in price units: 1.248
as a % of price   : 0.95%


,high,low,true_range,atr_14
time,,,,
2023-12-05,133.4956,133.2470,1.1064,1.3303
2023-12-06,132.7272,132.0968,1.2745,1.3242
2023-12-07,131.7028,131.6134,0.7985,1.2478


`pd.concat([...], axis=1).max(axis=1)` puts the three candidate ranges side by side as three columns and takes the row-wise maximum, the vectorized way to express "the largest of these three", with no loop. That is the Module 1 mindset applied to pandas.

### 6.6 Momentum and rolling extremes

Two simple ones that carry a lot of weight in real strategies.

**Momentum** is just price now minus price `n` bars ago (`self.mom(symbol, n)`); the percentage version, `self.momp`, is more common in cross-sectional work because it is comparable across assets.

**Rolling max/min** (`self.max(symbol, n)`, `self.min(symbol, n)`) define breakout levels, the classic Donchian channel.

In [13]:
mom_60 = px - px.shift(60)                 # absolute momentum, price units
momp_60 = px.pct_change(60)                # percentage momentum, comparable across assets

donchian_hi = px.rolling(20).max()
donchian_lo = px.rolling(20).min()

pd.DataFrame({
    "close": px,
    "mom_60": mom_60,
    "momp_60": momp_60,
    "20d_high": donchian_hi,
    "20d_low": donchian_lo,
}).tail(3)

,close,mom_60,momp_60,20d_high,20d_low
time,,,,,
2023-12-05,133.3713,7.8498,0.0625,135.0705,131.3947
2023-12-06,132.4120,5.8330,0.0461,134.3534,131.3947
2023-12-07,131.6581,6.6918,0.0535,134.3534,131.3947


### ✏️ Your turn: build the indicator set

Using `px` (the close series) and `bars`, build these four columns into a DataFrame called `ind`:

| column | definition |
|---|---|
| `sma_10` | 10-bar simple moving average |
| `ema_10` | 10-bar EMA, **`adjust=False`** |
| `rsi_7` | 7-bar RSI using the **simple** (rolling-mean) definition |
| `atr_10` | 10-bar ATR from the true range |

You may reuse the `rsi_simple` function defined above.

In [14]:
prev_c = bars["close"].shift(1)
true_range = pd.concat([
    bars["high"] - bars["low"],
    (bars["high"] - prev_c).abs(),
    (bars["low"] - prev_c).abs(),
], axis=1).max(axis=1)

ind = pd.DataFrame({
    "sma_10": px.rolling(10).mean(),
    "ema_10": px.ewm(span=10, adjust=False).mean(),
    "rsi_7": rsi_simple(px, 7),
    "atr_10": true_range.rolling(10).mean(),
})

ind.tail(3)

,sma_10,ema_10,rsi_7,atr_10
time,,,,
2023-12-05,133.2581,133.3974,49.2122,1.3379
2023-12-06,133.3269,133.2182,43.8289,1.4370
2023-12-07,133.1821,132.9345,47.4065,1.3732


In [15]:
_pc = bars["close"].shift(1)
_tr = pd.concat([bars["high"] - bars["low"],
                 (bars["high"] - _pc).abs(),
                 (bars["low"] - _pc).abs()], axis=1).max(axis=1)
_exp = pd.DataFrame({
    "sma_10": px.rolling(10).mean(),
    "ema_10": px.ewm(span=10, adjust=False).mean(),
    "rsi_7": rsi_simple(px, 7),
    "atr_10": _tr.rolling(10).mean(),
})
assert list(ind.columns) == ["sma_10", "ema_10", "rsi_7", "atr_10"], \
    f"columns should be sma_10, ema_10, rsi_7, atr_10 - got {list(ind.columns)}"
for col in _exp.columns:
    a, b = ind[col].dropna(), _exp[col].dropna()
    assert len(a) == len(b), f"{col}: expected {len(b)} non-NaN values, got {len(a)}"
    assert np.allclose(a.values, b.values), (
        f"{col} does not match" +
        (" - remember adjust=False on the EMA" if col == "ema_10" else ""))
print("✅ Correct!  Latest ATR is", round(float(ind['atr_10'].iloc[-1]), 3),
      "price units, RSI is", round(float(ind['rsi_7'].iloc[-1]), 1))

✅ Correct!  Latest ATR is 1.373 price units, RSI is 47.4


## 7. From indicator to signal

Now the design half. Almost every systematic equity strategy is one of four patterns, or a combination of them.

| Pattern | Idea | Typical indicators |
|---|---|---|
| **Crossover** | fast average crosses slow average → trend change | 2 SMAs / EMAs, MACD |
| **Threshold** | a bounded oscillator hits an extreme → fade it | RSI, %B, z-score |
| **Breakout** | price exceeds its recent range → trend continues | rolling max/min |
| **Filter** | only trade when a regime condition holds | 200-day MA, volatility, ATR |

Crossover and breakout are **trend following**: they buy strength. Threshold is usually **mean reversion**: it buys weakness. They tend to work in opposite market conditions, which is why combining them (or filtering one by the other) is so common.

### 7.1 Crossover

The signal is the *sign* of `fast − slow`. Expressing it as +1 / −1 gives you a position series you can multiply returns by.

In [16]:
fast = px.rolling(20).mean()
slow = px.rolling(50).mean()

signal = np.sign(fast - slow)               # +1 fast above slow, -1 below, 0 equal
signal = signal.where(slow.notna())         # undefined until the slow MA exists

crosses = signal.diff().abs() > 0           # a change of sign = a crossover event
print("crossover events:", int(crosses.sum()))
print("days long :", int((signal == 1).sum()))
print("days short:", int((signal == -1).sum()))

crossover events: 13
days long : 302
days short: 153


`np.sign` maps positives to +1, negatives to −1 and zero to 0: the whole crossover rule in one vectorized call, no loop and no `if`. `.where(slow.notna())` blanks out the warm-up period, which is the pandas equivalent of the `is_ready` guard.

Here are the actual crossover dates, which is what you would eyeball on a chart:

In [17]:
events = signal[crosses]
summary = pd.DataFrame({
    "date": [d.date() for d in events.index],
    "new_signal": events.values,
    "close": [round(float(px.loc[d]), 2) for d in events.index],
})
summary.head(8)

,date,new_signal,close
0,2022-03-25,1.0,104.11
1,2022-08-09,-1.0,120.23
2,2022-08-19,1.0,128.78
3,2022-10-05,-1.0,120.65
4,2022-11-10,1.0,121.72
5,2022-11-21,-1.0,114.58
6,2023-01-04,1.0,127.96
7,2023-02-20,-1.0,121.73


### 7.2 Threshold (mean reversion)

Buy when the oscillator is extreme-low, sell when extreme-high. The subtlety: RSI spends most of its time in the middle, where you want **no position** rather than a position, so the naive `np.sign` approach does not apply.

The idiom is to build the signal in three states and then **hold** it until the opposite condition fires.

In [18]:
rsi = rsi_wilder(px, 14)

raw = pd.Series(np.nan, index=px.index)
raw[rsi < 30] = 1.0                # oversold  -> go long
raw[rsi > 70] = -1.0               # overbought -> go short
raw[(rsi > 45) & (rsi < 55)] = 0.0  # back to neutral -> flatten

position = raw.ffill().fillna(0.0)   # hold the last decision until a new one fires

print("days long   :", int((position == 1).sum()))
print("days short  :", int((position == -1).sum()))
print("days flat   :", int((position == 0).sum()))

days long   : 5
days short  : 99
days flat   : 400


That `ffill` is doing real work. Without it you would hold a position on the single day RSI dipped below 30 and then go flat immediately, which is not what "buy oversold" means. **A signal is a state, not an event**. The `ffill` is what turns events into states.

This is one place where the pandas version and the algorithm version genuinely diverge in *feel*: in `on_data` the state is simply whatever `self.portfolio[symbol].invested` says, and you only act when you want to change it.

### ✏️ Your turn: a held mean-reversion signal

Build `mr_position`, a long-only mean-reversion state from a **10-bar Wilder RSI** (use the `rsi_wilder` function above):

- go **long** (`1.0`) when RSI drops below **25**
- go **flat** (`0.0`) when RSI rises back above **55**
- **hold** the last decision on every other day
- start flat, so there should be no `NaN` left

Then set `days_long` to the number of days the state is `1.0`.

*Hint: assign the two states into a `NaN`-filled Series, then `ffill().fillna(0.0)`.*

In [19]:
rsi_10 = rsi_wilder(px, 10)

raw_state = pd.Series(np.nan, index=px.index)
raw_state[rsi_10 < 25] = 1.0
raw_state[rsi_10 > 55] = 0.0

mr_position = raw_state.ffill().fillna(0.0)

days_long = int((mr_position == 1.0).sum())

print("days long:", days_long)
mr_position.tail(3)

days long: 22


time
2023-12-05    0.0
2023-12-06    0.0
2023-12-07    0.0
Freq: B, dtype: float64

In [20]:
_r10 = rsi_wilder(px, 10)
_raw = pd.Series(np.nan, index=px.index)
_raw[_r10 < 25] = 1.0
_raw[_r10 > 55] = 0.0
_pos = _raw.ffill().fillna(0.0)
assert np.allclose(rsi_10.dropna().values, _r10.dropna().values), \
    "rsi_10 should be rsi_wilder(px, 10)"
assert not mr_position.isna().any(), "mr_position should have no NaN - ffill then fillna(0.0)"
assert set(np.unique(mr_position.values)) <= {0.0, 1.0}, "mr_position should only be 0.0 or 1.0"
assert np.allclose(mr_position.values, _pos.values), (
    "state values are off - remember to HOLD the position between triggers "
    "rather than only marking the trigger days")
assert days_long == int((_pos == 1.0).sum()), f"expected {int((_pos == 1.0).sum())} long days"
print("✅ Correct!  Long on", days_long, "of 504 days;",
      "entries triggered:", int((_r10 < 25).sum()))

✅ Correct!  Long on 22 of 504 days; entries triggered: 2


### 7.3 Breakout

Buy when price closes above its own `n`-day high. Because today's close is part of today's rolling max, you must compare against the max **excluding today**, or the signal can never fire.

In [21]:
hi_20 = px.rolling(20).max().shift(1)     # highest close of the PREVIOUS 20 bars
lo_20 = px.rolling(20).min().shift(1)

breakout = pd.Series(0.0, index=px.index)
breakout[px > hi_20] = 1.0
breakout[px < lo_20] = -1.0

print("upside breakouts  :", int((breakout == 1).sum()))
print("downside breakouts:", int((breakout == -1).sum()))
print()
print("Without the .shift(1), price can never exceed its own rolling max:")
print("  breakouts detected:", int((px > px.rolling(20).max()).sum()))

upside breakouts  : 80
downside breakouts: 29

Without the .shift(1), price can never exceed its own rolling max:
  breakouts detected: 0


That last line is the point: **0 breakouts**. Today's close is included in today's 20-day maximum, so `px > px.rolling(20).max()` is never true. The `.shift(1)` is not a stylistic choice, it is the difference between a working signal and a dead one.

### 7.4 Regime filter

A filter does not generate trades; it *vetoes* them. The classic is: only take long signals when price is above its 200-day moving average.

In [22]:
trend_ok = px > px.rolling(200).mean()

raw_signal = np.sign(fast - slow).where(slow.notna()).fillna(0.0)
filtered = raw_signal.where(raw_signal < 0, raw_signal * trend_ok.astype(float))

print("long days before filter:", int((raw_signal == 1).sum()))
print("long days after filter :", int((filtered == 1).sum()))
print("short days unchanged   :", int((filtered == -1).sum()))

long days before filter: 302
long days after filter : 154
short days unchanged   : 153


The filter removed long signals that fired while price was below its 200-day average, exactly the whipsaw trades that hurt trend followers in choppy markets. Whether that improves the strategy is an empirical question, and Module 6 gives you the tools to answer it honestly.

### ✏️ Your turn: build a crossover signal

Using `px`:

1. Build `f10` and `s30`, the 10-bar and 30-bar simple moving averages.
2. Build `cross_signal`: `+1` where `f10 > s30`, `-1` where `f10 < s30`, and `NaN` wherever `s30` is `NaN` (the warm-up period). Use `np.sign` and `.where`.
3. Set `n_crossovers` to the number of times the signal *changes*.

In [23]:
f10 = px.rolling(10).mean()
s30 = px.rolling(30).mean()

cross_signal = np.sign(f10 - s30).where(s30.notna())

n_crossovers = int((cross_signal.diff().abs() > 0).sum())

print("crossovers:", n_crossovers)
cross_signal.dropna().head(3)

crossovers: 16


time
2022-02-11   -1.0
2022-02-14   -1.0
2022-02-15   -1.0
Freq: B, Name: close, dtype: float64

In [24]:
_f, _s = px.rolling(10).mean(), px.rolling(30).mean()
_sig = np.sign(_f - _s).where(_s.notna())
_n = int((_sig.diff().abs() > 0).sum())
assert np.allclose(f10.dropna().values, _f.dropna().values), "f10 should be the 10-bar SMA"
assert np.allclose(s30.dropna().values, _s.dropna().values), "s30 should be the 30-bar SMA"
assert cross_signal.isna().sum() == 29, \
    f"expected 29 NaN warm-up values, got {cross_signal.isna().sum()}"
assert np.allclose(cross_signal.dropna().values, _sig.dropna().values), \
    "signal values are off - use np.sign(f10 - s30)"
assert n_crossovers == _n, f"expected {_n} crossovers, got {n_crossovers}"
print("✅ Correct!", n_crossovers, "crossovers across the four regimes")

✅ Correct! 16 crossovers across the four regimes


### ✏️ Your turn: a filtered breakout signal

Build `bo_signal`, a 55-day breakout with a trend filter:

1. `hi_55` = the highest close of the **previous** 55 bars (remember the `.shift(1)`).
2. `ma_100` = the 100-bar simple moving average.
3. `bo_signal` = `1.0` on days where price closes above `hi_55` **and** price is above `ma_100`; `0.0` everywhere else. It should be a float Series with no `NaN`.

Then set `n_signals` to the number of `1.0` days.

In [25]:
hi_55 = px.rolling(55).max().shift(1)
ma_100 = px.rolling(100).mean()

bo_signal = ((px > hi_55) & (px > ma_100)).astype(float)

n_signals = int(bo_signal.sum())

print("breakout days passing the filter:", n_signals)

breakout days passing the filter: 39


In [26]:
_hi = px.rolling(55).max().shift(1)
_ma = px.rolling(100).mean()
_bo = ((px > _hi) & (px > _ma)).astype(float)
assert np.allclose(hi_55.dropna().values, _hi.dropna().values), \
    "hi_55 should be rolling(55).max().shift(1) - without the shift it can never be exceeded"
assert np.allclose(ma_100.dropna().values, _ma.dropna().values), "ma_100 should be the 100-bar SMA"
assert not bo_signal.isna().any(), "bo_signal should contain no NaN - use .astype(float) on the boolean"
assert set(np.unique(bo_signal.values)) <= {0.0, 1.0}, "bo_signal should only contain 0.0 and 1.0"
assert np.allclose(bo_signal.values, _bo.values), "bo_signal values are off"
assert n_signals == int(_bo.sum()), f"expected {int(_bo.sum())} signal days, got {n_signals}"
print("✅ Correct!", n_signals, "of 504 days were filtered breakouts")

✅ Correct! 39 of 504 days were filtered breakouts


## 8. The one bug that invalidates everything: acting on today's close

Here is the mistake that makes a beginner's backtest look brilliant.

You compute a signal from today's closing price and then apply it to *today's* return. But you could not have known today's close until the market closed, at which point today's return has already happened. You have traded on information you did not have.

The fix is one character of code and a large amount of discipline: **shift the signal forward by one bar** before applying it to returns.

In [27]:
ret = px.pct_change()
sig = np.sign(fast - slow).where(slow.notna()).fillna(0.0)

wrong = (sig * ret).dropna()                 # signal from today's close, today's return
right = (sig.shift(1) * ret).dropna()        # act on the NEXT bar

comparison = pd.DataFrame({
    "total_return": [float((1 + wrong).prod() - 1), float((1 + right).prod() - 1)],
    "ann_sharpe": [float(wrong.mean() / wrong.std() * np.sqrt(252)),
                   float(right.mean() / right.std() * np.sqrt(252))],
}, index=["look-ahead (WRONG)", "shifted (correct)"])
comparison

,total_return,ann_sharpe
look-ahead (WRONG),-0.3438,-1.2375
shifted (correct),-0.2842,-0.9648


Two versions of the same strategy, one number of difference in the code, and a materially different result. The look-ahead version is not a better strategy; it is a strategy that cheats.

> 🧠 **Why `on_data` protects you.** In an algorithm this bug is much harder to write, because `on_data` is *called with* today's bar and any order you place fills at the next available price. The event-driven model enforces causality that a DataFrame does not. This is the single strongest argument for validating research ideas in an actual backtest rather than trusting a notebook.

The QuantConnect version of the crossover, with all of Module 3's pillars in place:

In [ ]:
# 🔵 QC cell, the crossover as a complete algorithm
class MovingAverageCrossover(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)

        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        self.fast = self.sma(self.symbol, 20, Resolution.DAILY)
        self.slow = self.sma(self.symbol, 50, Resolution.DAILY)

        self.set_warm_up(60, Resolution.DAILY)

    def on_data(self, data: Slice):
        if self.is_warming_up:
            return
        if not (self.fast.is_ready and self.slow.is_ready):
            return
        if not data.bars.contains_key(self.symbol):
            return

        fast_above = self.fast.current.value > self.slow.current.value
        invested = self.portfolio[self.symbol].invested

        # Only ACT on a change of state. Calling set_holdings every bar would
        # place redundant orders and pay spread for nothing.
        if fast_above and not invested:
            self.set_holdings(self.symbol, 1.0)
            self.debug(f"{self.time.date()} LONG at {data.bars[self.symbol].close:.2f}")
        elif not fast_above and invested:
            self.liquidate(self.symbol)
            self.debug(f"{self.time.date()} FLAT at {data.bars[self.symbol].close:.2f}")

Read the `on_data` guard stack once more, because this ordering is a template you should internalise:

1. `is_warming_up`: is this even a real bar?
2. `is_ready`: do my indicators mean anything yet?
3. `contains_key`: did data actually arrive for my symbol?
4. compute the signal
5. compare against **current state** (`invested`) and only act on a change

Every robust QuantConnect algorithm you write this year will have those five steps in that order.

### ✏️ Your turn: shift the signal, measure the difference

Using `px` and the `cross_signal` you built earlier:

1. Build `daily_ret`, daily simple returns of `px`.
2. Build `strat_ret`, the **correctly shifted** strategy returns: yesterday's signal times today's return, `NaN`s dropped. Fill the signal's warm-up `NaN`s with `0.0` before shifting.
3. Set `total_return` to the strategy's total compounded return over the whole sample.

In [28]:
daily_ret = px.pct_change()

strat_ret = (cross_signal.fillna(0.0).shift(1) * daily_ret).dropna()

total_return = (1 + strat_ret).prod() - 1

print(f"total return: {total_return:.2%}")

total return: -3.17%


In [29]:
_dr = px.pct_change()
_sr = (cross_signal.fillna(0.0).shift(1) * _dr).dropna()
_tot = float((1 + _sr).prod() - 1)
_wrong = (cross_signal.fillna(0.0) * _dr).dropna()
assert np.allclose(daily_ret.dropna().values, _dr.dropna().values), "daily_ret should be px.pct_change()"
assert not np.allclose(strat_ret.values, _wrong.values), \
    "that looks unshifted - you are trading on today's close, which is look-ahead bias"
assert len(strat_ret) == len(_sr), f"expected {len(_sr)} rows, got {len(strat_ret)}"
assert np.allclose(strat_ret.values, _sr.values), "strat_ret values are off"
assert np.isclose(float(total_return), _tot), "total_return should be (1 + strat_ret).prod() - 1"
print(f"✅ Correct!  Shifted strategy total return: {_tot:.2%}",
      f"(look-ahead version would have claimed {float((1 + _wrong).prod() - 1):.2%})")

✅ Correct!  Shifted strategy total return: -3.17% (look-ahead version would have claimed 7.07%)


## 9. Combining signals

Real strategies stack a few of these patterns. Three combination rules cover most cases:

| Rule | Code | Use when |
|---|---|---|
| **AND** (both must agree) | `sig_a * sig_b` for 0/1 signals | you want fewer, higher-conviction trades |
| **VETO** (filter) | `sig * condition.astype(float)` | one input is a regime gate, not a signal |
| **VOTE** (average) | `(a + b + c) / 3` | several weak signals, none trusted alone |

The AND rule shrinks your trade count fast, two independent signals that each fire 30% of the time agree only about 9% of the time. That is often a good trade-off, but check that you are left with enough trades to draw a conclusion from. Module 6 covers how many trades you need before a backtest result means anything.

In [30]:
trend_up = (px > px.rolling(100).mean()).astype(float)
mom_up = (px.pct_change(20) > 0).astype(float)
not_stretched = (rsi_wilder(px, 14) < 70).astype(float)

combined = trend_up * mom_up * not_stretched

vote = (trend_up + mom_up + not_stretched) / 3

pd.DataFrame({
    "days_active": [int(trend_up.sum()), int(mom_up.sum()), int(not_stretched.sum()),
                    int(combined.sum()), int((vote > 0.5).sum())],
}, index=["trend_up", "mom_up", "not_stretched", "AND of all three", "VOTE > 0.5"])

,days_active
trend_up,301
mom_up,290
not_stretched,458
AND of all three,193
VOTE > 0.5,372


The AND row is much smaller than any individual input, and the VOTE row sits in between. Which you want depends on whether you would rather be right more often or be invested more often.

### ✏️ Your turn: combine three conditions

Build three 0/1 float condition Series from `px`, then combine them:

| name | condition |
|---|---|
| `above_50` | close is above its 50-bar SMA |
| `pos_mom` | 10-bar percentage momentum is positive |
| `low_vol` | 20-bar rolling std of daily returns is **below** its own median |

Then build `and_signal` (all three must hold) and `vote_signal` (`1.0` when at least two of the three hold, else `0.0`), and set `and_days` / `vote_days` to how many days each fires.

In [31]:
daily_r = px.pct_change()
vol20 = daily_r.rolling(20).std()

above_50 = (px > px.rolling(50).mean()).astype(float)
pos_mom = (px.pct_change(10) > 0).astype(float)
low_vol = (vol20 < vol20.median()).astype(float)

and_signal = above_50 * pos_mom * low_vol
vote_signal = ((above_50 + pos_mom + low_vol) >= 2).astype(float)

and_days = int(and_signal.sum())
vote_days = int(vote_signal.sum())

print("AND:", and_days, "| VOTE:", vote_days)

AND: 137 | VOTE: 296


In [32]:
_dr = px.pct_change()
_v20 = _dr.rolling(20).std()
_a = (px > px.rolling(50).mean()).astype(float)
_p = (px.pct_change(10) > 0).astype(float)
_l = (_v20 < _v20.median()).astype(float)
_and = _a * _p * _l
_vote = ((_a + _p + _l) >= 2).astype(float)
for name, got, exp in [("above_50", above_50, _a), ("pos_mom", pos_mom, _p), ("low_vol", low_vol, _l)]:
    assert set(np.unique(got.values)) <= {0.0, 1.0}, f"{name} should be a 0.0/1.0 float Series"
    assert np.allclose(got.values, exp.values), f"{name} is off"
assert np.allclose(and_signal.values, _and.values), \
    "and_signal should be 1.0 only when all three conditions hold"
assert np.allclose(vote_signal.values, _vote.values), \
    "vote_signal should be 1.0 when at least two of the three hold"
assert and_days == int(_and.sum()) and vote_days == int(_vote.sum()), "the day counts are off"
assert and_days < vote_days, "the AND rule should fire on strictly fewer days than the VOTE rule"
print(f"✅ Correct!  AND fires {and_days} days, VOTE fires {vote_days} -",
      f"the AND rule is {vote_days - and_days} days more selective")

✅ Correct!  AND fires 137 days, VOTE fires 296 - the AND rule is 159 days more selective


## 🏁 Mini-challenge: a complete signal pipeline

Wrap everything into one reusable function. You will paste variations of this into every research notebook for the rest of the curriculum.

### ✏️ Your turn: a signal pipeline function

Write `build_strategy(prices, fast=20, slow=50, trend=200)` that returns a DataFrame with these five columns, in this order:

| column | definition |
|---|---|
| `close` | the input price series |
| `fast_ma` | `fast`-bar SMA |
| `slow_ma` | `slow`-bar SMA |
| `signal` | `1.0` when `fast_ma > slow_ma` **and** `close > trend`-bar SMA, else `0.0` |
| `strategy_ret` | `signal` shifted one bar, times the daily return |

The index must match `prices`. `signal` must contain no `NaN`. `strategy_ret` will have one leading `NaN` from `pct_change`. Leave it.

In [33]:
def build_strategy(prices, fast=20, slow=50, trend=200):
    fast_ma = prices.rolling(fast).mean()
    slow_ma = prices.rolling(slow).mean()
    trend_ma = prices.rolling(trend).mean()

    signal = ((fast_ma > slow_ma) & (prices > trend_ma)).astype(float)

    strategy_ret = signal.shift(1) * prices.pct_change()

    return pd.DataFrame({
        "close": prices,
        "fast_ma": fast_ma,
        "slow_ma": slow_ma,
        "signal": signal,
        "strategy_ret": strategy_ret,
    })


result = build_strategy(px)
result.tail(3)

,close,fast_ma,slow_ma,signal,strategy_ret
time,,,,,
2023-12-05,133.3713,132.9264,132.1692,1.0,-0.0073
2023-12-06,132.4120,132.7935,132.2839,1.0,-0.0072
2023-12-07,131.6581,132.7284,132.3592,1.0,-0.0057


In [34]:
out = build_strategy(px)
_f, _s, _t = px.rolling(20).mean(), px.rolling(50).mean(), px.rolling(200).mean()
_sig = ((_f > _s) & (px > _t)).astype(float)
_sr = _sig.shift(1) * px.pct_change()

assert list(out.columns) == ["close", "fast_ma", "slow_ma", "signal", "strategy_ret"], \
    f"columns must be in the stated order, got {list(out.columns)}"
assert out.index.equals(px.index), "the index should match the input prices"
assert not out["signal"].isna().any(), "signal should have no NaN - use .astype(float) on the boolean"
assert set(np.unique(out["signal"].values)) <= {0.0, 1.0}, "signal should only be 0.0 or 1.0"
assert np.allclose(out["signal"].values, _sig.values), \
    "signal is off - it needs BOTH the crossover and the trend filter"
assert np.allclose(out["strategy_ret"].dropna().values, _sr.dropna().values), \
    "strategy_ret is off - shift the signal by one bar before multiplying"
assert out["strategy_ret"].isna().sum() == 1, \
    "expected exactly one leading NaN in strategy_ret"

# the parameters must actually be wired through
alt = build_strategy(px, fast=5, slow=15, trend=50)
assert not np.allclose(alt["signal"].values, out["signal"].values), \
    "changing fast/slow/trend should change the signal - are the parameters used?"

active = float(out["signal"].mean())
tot = float((1 + out["strategy_ret"].fillna(0)).prod() - 1)
print(f"✅ Correct!  Invested {active:.1%} of days, total return {tot:.2%}")

✅ Correct!  Invested 30.6% of days, total return -18.03%


## Cheat sheet

**Creating indicators (in `initialize`)**

| Task | Code |
|---|---|
| SMA / EMA | `self.sma(sym, 20)` · `self.ema(sym, 20)` |
| RSI | `self.rsi(sym, 14)` |
| Bollinger Bands | `self.bb(sym, 20, 2)` |
| ATR | `self.atr(sym, 14)` |
| MACD | `self.macd(sym, 12, 26, 9)` |
| Std dev / Momentum | `self.std(sym, 20)` · `self.mom(sym, 60)` · `self.momp(sym, 60)` |
| Rolling extremes | `self.max(sym, 20)` · `self.min(sym, 20)` |
| Manual indicator | `SimpleMovingAverage(20)` then `ind.update(self.time, value)` |
| Let QC feed a manual one | `self.register_indicator(sym, ind, Resolution.DAILY)` |
| Warm up | `self.set_warm_up(60, Resolution.DAILY)` |

**Reading them (in `on_data`)**

| Task | Code |
|---|---|
| Value | `self.sma_20.current.value` |
| Ready? | `self.sma_20.is_ready` |
| Still warming up? | `self.is_warming_up` |
| Band values | `self.bb_20.upper_band.current.value` |
| Current time | `self.time` |

**pandas equivalents (for research)**

| Indicator | pandas |
|---|---|
| SMA | `px.rolling(n).mean()` |
| EMA | `px.ewm(span=n, adjust=False).mean()` |
| Std dev | `px.rolling(n).std(ddof=0)` |
| Bollinger | `mid ± k * px.rolling(n).std(ddof=0)` |
| RSI (Wilder) | gains/losses → `ewm(alpha=1/n, adjust=False)` → `100 − 100/(1+RS)` |
| True range | row-wise max of `h−l`, `|h−c₋₁|`, `|l−c₋₁|` |
| ATR | `tr.rolling(n).mean()` |
| Momentum | `px.diff(n)` · `px.pct_change(n)` |
| Donchian high | `px.rolling(n).max().shift(1)` |

**Signal patterns**

| Pattern | pandas |
|---|---|
| Crossover | `np.sign(fast - slow)` |
| Threshold with hold | set states, then `.ffill()` |
| Breakout | `px > px.rolling(n).max().shift(1)` |
| Filter | `signal * condition.astype(float)` |
| **No look-ahead** | `signal.shift(1) * returns` |

## Stretch goals

Bring these to the next meeting:

1. **Parameter sweep.** Loop `fast` over `[5, 10, 20]` and `slow` over `[30, 50, 100]`, run `build_strategy` for all nine pairs, and tabulate total return. Do the best parameters look robust, or is one cell suspiciously better than its neighbours? (That is the overfitting question Module 6 makes formal.)
2. **SMA versus EMA.** Rerun the crossover with EMAs instead of SMAs. Does reacting faster help or hurt across the four regimes in our data?
3. **Regime attribution.** Split `strategy_ret` into the four 126-day regimes and compute the return of each. Confirm the trend follower loses money in the choppy regime.
4. **Add a short side.** Change `build_strategy` so `signal` is `-1.0` when the fast MA is below the slow MA *and* price is below the trend MA. Does adding shorts improve the result, and does it change the drawdown?
5. **Match QuantConnect exactly.** Run `qb.indicator(RelativeStrengthIndex(14), spy, 500, Resolution.DAILY)` in a QC research notebook and compare it to both `rsi_simple` and `rsi_wilder`. Which one matches, and by how much does the other differ?

## What's next

**Module 6: Backtesting Mechanics & Performance Analytics** answers the question this module keeps deferring: *is this signal any good?* You will learn what QuantConnect's backtest actually simulates (fills, slippage, fees, cash settlement), how to read every statistic on the results page, and the discipline (out-of-sample testing, parameter stability, trade counts) that separates a result you can trust from one you have fooled yourself with.

**Official docs:**
- [Indicators overview](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/key-concepts)
- [Supported indicators](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/supported-indicators)
- [Manual indicators](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/manual-indicators)
- [Warm-up periods](https://www.quantconnect.com/docs/v2/writing-algorithms/historical-data/warm-up-periods)
- [Indicators in research](https://www.quantconnect.com/docs/v2/research-environment/indicators)

*MAT Education · QuantConnect Core · Module 5.*